# NLP Coursework — Orchestrator Notebook

This notebook is a **thin orchestrator**: every piece of logic lives in the
tested, typed `nlp_toolkit` package under [`src/`](../src/nlp_toolkit). Each
cell below simply wires inputs to toolkit functions and displays results, so the
notebook stays readable and the logic stays reusable and unit-tested.

**Setup once (with [uv](https://docs.astral.sh/uv/), from the repo root in a terminal):**

```bash
uv sync --extra dev                              # create .venv and install all packages
uv run python scripts/download_resources.py      # NLTK data + spaCy model
```

Then select this project's `.venv` as the notebook kernel (the
**Python (nlp-toolkit)** kernel) and run the cells below. See the
[README](../README.md#installation) for details and a pip fallback.

In [ ]:
# --- One-time environment setup --------------------------------------------
# Install packages with uv (run these in a terminal from the repo root):
#     uv sync --extra dev
#     uv run python scripts/download_resources.py   # NLTK data + spaCy model
#
# uv installs everything into this project's .venv; select that interpreter
# (the "Python (nlp-toolkit)" kernel) before running the cells below.
#
# Prefer to provision from inside the notebook instead? Uncomment:
# !uv run python ../scripts/download_resources.py

In [1]:
# --- Bootstrap -------------------------------------------------------------
import sys
from pathlib import Path

# Make the src/ layout importable when running from notebooks/.
SRC = Path.cwd().parent / "src"
if SRC.is_dir() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from nltk.corpus import gutenberg

from nlp_toolkit.config import get_settings
from nlp_toolkit.logging_config import configure_logging
from nlp_toolkit.resources import ensure_nltk_data
from nlp_toolkit.utils import seed_everything

from nlp_toolkit.preprocessing import (
    lemmatize_tokens, normalize_words, preprocess_for_tfidf, remove_stopwords, stem_tokens,
)
from nlp_toolkit.analysis import (
    compute_corpus_stats, count_occurrences, count_words_matching, count_words_of_length,
    gutenberg_word_counts, most_common_words, word_frequencies, word_set_difference,
)
from nlp_toolkit.analysis.regex_search import count_words_ending_with
from nlp_toolkit.data import load_lexicon, load_reviews, load_text_file
from nlp_toolkit.features import top_tfidf_words
from nlp_toolkit.nlp import count_entities_by_label
from nlp_toolkit.sentiment import LexiconSentimentAnalyzer, SentimentLexicon

configure_logging()
seed_everything()                # reproducibility
settings = get_settings()
ensure_nltk_data(settings)       # download NLTK data only if missing
settings.data_dir

2026-06-26 11:08:34 | INFO     | nlp_toolkit.resources | Downloading NLTK package: wordnet
2026-06-26 11:08:34 | INFO     | nlp_toolkit.resources | Downloading NLTK package: omw-1.4


WindowsPath('C:/Users/Erick/Documents/vscode/nlp/notebooks/data')

## 1) Estatísticas das peças de Shakespeare

Para *Caesar*, *Hamlet* e *Macbeth*: total de palavras, sentenças, palavras
únicas, repetidas e média de palavras por sentença.

In [2]:
PLAYS = {
    "Caesar": "shakespeare-caesar.txt",
    "Hamlet": "shakespeare-hamlet.txt",
    "Macbeth": "shakespeare-macbeth.txt",
}

play_stats = {
    name: compute_corpus_stats(gutenberg.words(fid), gutenberg.sents(fid))
    for name, fid in PLAYS.items()
}

for name, stats in play_stats.items():
    print(f"{name}: words={stats.total_words}, sentences={stats.total_sentences}, "
          f"unique={stats.unique_words}, repeated={stats.repeated_words}, "
          f"words/sentence={stats.words_per_sentence:.2f}")

Caesar: words=20802, sentences=2163, unique=3014, repeated=17788, words/sentence=9.62
Hamlet: words=30266, sentences=3106, unique=4699, repeated=25567, words/sentence=9.74
Macbeth: words=18272, sentences=1907, unique=3447, repeated=14825, words/sentence=9.58


## 2) Análise do corpus Gutenberg

Total por documento, maior/menor documento, média de sentenças por palavra,
distribuição de frequência de *Macbeth*, top-5 palavras e diferença de
vocabulário entre *Caesar* e *Hamlet*.

In [3]:
# a) palavras por documento
counts = gutenberg_word_counts()
for fid, n in counts.items():
    print(f"{fid}: {n}")

# b/c) maior e menor documento
print("\nLargest :", max(counts, key=counts.get), counts[max(counts, key=counts.get)])
print("Smallest:", min(counts, key=counts.get), counts[min(counts, key=counts.get)])

# d) média de sentenças por palavra no corpus inteiro
total_words = sum(counts.values())
total_sents = sum(len(gutenberg.sents(fid)) for fid in counts)
print(f"\nSentences per word: {total_sents / total_words:.6f}")

austen-emma.txt: 161600
austen-persuasion.txt: 84121
austen-sense.txt: 120733
bible-kjv.txt: 791842
blake-poems.txt: 6934
bryant-stories.txt: 46611
burgess-busterbrown.txt: 16327
carroll-alice.txt: 27333
chesterton-ball.txt: 82682
chesterton-brown.txt: 73286
chesterton-thursday.txt: 58724
edgeworth-parents.txt: 170737
melville-moby_dick.txt: 218361
milton-paradise.txt: 80493
shakespeare-caesar.txt: 20802
shakespeare-hamlet.txt: 30266
shakespeare-macbeth.txt: 18272
whitman-leaves.txt: 126276

Largest : bible-kjv.txt 791842
Smallest: blake-poems.txt 6934

Sentences per word: 0.046152


In [4]:
# e/f) distribuição de frequência de Macbeth e top-5
macbeth_words = normalize_words(gutenberg.words("shakespeare-macbeth.txt"))
print("Top 10:", word_frequencies(macbeth_words).most_common(10))
print("Top 5 :", most_common_words(macbeth_words, settings.top_n))

# g) diferença de vocabulário Caesar vs Hamlet
caesar_words = normalize_words(gutenberg.words("shakespeare-caesar.txt"))
hamlet_words = normalize_words(gutenberg.words("shakespeare-hamlet.txt"))
print("\nIn Caesar not Hamlet:", sorted(word_set_difference(caesar_words, hamlet_words))[:10])
print("In Hamlet not Caesar:", sorted(word_set_difference(hamlet_words, caesar_words))[:10])

Top 10: [('the', 650), ('and', 546), ('to', 384), ('i', 348), ('of', 338), ('a', 241), ('that', 238), ('d', 224), ('you', 206), ('my', 203)]
Top 5 : [('the', 650), ('and', 546), ('to', 384), ('i', 348), ('of', 338)]

In Caesar not Hamlet: ['abide', 'abler', 'abridg', 'absence', 'accents', 'accoutred', 'acquainted', 'acrosse', 'adde', 'added']
In Hamlet not Caesar: ['abhominably', 'abhorred', 'abilitie', 'aboord', 'abridgements', 'absolute', 'abstinence', 'abstracts', 'absurd', 'abus']


## 3) Buscas com expressões regulares em *Caesar*

(a) palavras terminadas em "r"; (b) palavras com 5 letras; (c) ocorrências de
"err"; (d) palavras de 5+ letras contendo "are".

In [5]:
print("Words ending in 'r' :", count_words_matching(caesar_words, r"\w+r\b"))
print("Words with 5 letters:", count_words_of_length(caesar_words, 5))
print("'err' occurrences   :", count_occurrences(caesar_words, "err"))

# d) palavras de 5 letras contendo "are" (comportamento original preservado)
print("5-letter words with 'are':",
      sum(1 for w in caesar_words if len(w) == 5 and "are" in w.lower()))

Words ending in 'r' : 1323
Words with 5 letters: 2952
'err' occurrences   : 16
5-letter words with 'are': 111


## 4) Pré-processamento de *Hamlet*

(a) normalização; (b) stemming + lematização; (c) tokenização; (d) contagem de
adjetivos (tag `JJ`).

In [6]:
import nltk

# a) normalização
hamlet_norm = normalize_words(gutenberg.words("shakespeare-hamlet.txt"))
print("Normalized:", hamlet_norm[:10])

# b) stemming seguido de lematização (preserva o comportamento original)
hamlet_processed = lemmatize_tokens(stem_tokens(hamlet_norm))
print("Stem+Lemma:", hamlet_processed[:10])

# c) tokenização e d) contagem de adjetivos
tagged = nltk.pos_tag(hamlet_processed)
adjectives = sum(1 for _, tag in tagged if tag == "JJ")
print("Adjectives (JJ):", adjectives)

Normalized: ['the', 'tragedie', 'of', 'hamlet', 'by', 'william', 'shakespeare', 'actus', 'primus', 'scoena']
Stem+Lemma: ['the', 'tragedi', 'of', 'hamlet', 'by', 'william', 'shakespear', 'actu', 'primu', 'scoena']
Adjectives (JJ): 2878


## 5) Remoção de stopwords

Contagem de palavras restantes em *Caesar* e *Hamlet* após remover stopwords.

> **Correção de bug:** a versão original media `len()` de uma *string* (número
> de caracteres) ao invés do número de *palavras*. Aqui contamos tokens.

In [7]:
caesar_no_stop = remove_stopwords(caesar_words)
hamlet_no_stop = remove_stopwords(hamlet_words)

print(f"Caesar: {len(caesar_words)} -> {len(caesar_no_stop)} words")
print(f"Hamlet: {len(hamlet_words)} -> {len(hamlet_no_stop)} words")

Caesar: 20802 -> 11056 words
Hamlet: 30266 -> 15898 words


## 6) Reconhecimento de entidades nomeadas (spaCy)

Contagem de entidades `GPE`, `LOC` e `PERSON` em `apoloxi.txt` e
`french-revolution.txt`. O documento é processado **uma única vez** por arquivo.

In [8]:
docs = {
    "Apoloxi": load_text_file(settings.data_dir / "apoloxi.txt"),
    "French Revolution": load_text_file(settings.data_dir / "french-revolution.txt"),
}

for name, text in docs.items():
    labels = count_entities_by_label(text)
    print(f"{name}: GPE={labels['GPE']}, LOC={labels['LOC']}, PERSON={labels['PERSON']}")

RuntimeError: spaCy model 'en_core_web_sm' is not installed. Run: python -m spacy download en_core_web_sm

## 7) Análise de sentimento baseada em léxico

Polaridade das reviews usando `positive_words.csv` / `negative_words.csv`.
Lexicons carregados como conjuntos (busca O(1)).

In [9]:
lexicon = SentimentLexicon(
    positive=load_lexicon(settings.positive_words_path),
    negative=load_lexicon(settings.negative_words_path),
)
analyzer = LexiconSentimentAnalyzer(lexicon)

reviews = load_reviews(settings.reviews_path)
for review in reviews:
    print(f"{analyzer.polarity(review):+.3f}  {review[:80]}")

print(f"\nTotal polarity: {analyzer.corpus_polarity(reviews):.3f}")

2026-06-26 11:08:51 | INFO     | nlp_toolkit.data.loaders | Loaded 8 reviews from C:\Users\Erick\Documents\vscode\nlp\notebooks\data\reviews.csv


+0.600  This camera is perfect for an enthusiastic amateur photographer. The pictures ar
+0.875  I got my camera three days back, and although i had some experience with digital
+0.667  I love photography. I had an older camera that was simply a point and shoot came
+0.182  I bought coolpix 4300 two months after i had bought canon powershot s400. It was
+0.286  The other reviewers have clearly pointed all the good things about this camera, 
+0.000  Within a year, there are problems with my menu dial knob. It became stuck which 
+0.000  Got a "system error" problem 30 days after purchase. Made the camera totally ino
+0.455  I am an amateur photographer and here is a piece of advise to all the folks who 

Total polarity: 3.064


## 8) Palavras mais relevantes por TF-IDF

Top-5 palavras de cada documento após normalização, remoção de stopwords e
lematização.

In [10]:
TFIDF_TEXTS = [
    "austen-emma.txt", "bible-kjv.txt", "carroll-alice.txt",
    "melville-moby_dick.txt", "shakespeare-caesar.txt", "shakespeare-hamlet.txt",
]

documents = [preprocess_for_tfidf(gutenberg.raw(fid)) for fid in TFIDF_TEXTS]
top_words = top_tfidf_words(documents, labels=TFIDF_TEXTS, top_n=settings.top_n)

for label, words in top_words.items():
    print(f"{label}: {', '.join(words)}")

austen-emma.txt: emma, harriet, weston, elton, knightley
bible-kjv.txt: unto, shall, lord, thou, god
carroll-alice.txt: alice, said, hatter, little, gryphon
melville-moby_dick.txt: whale, ahab, one, ship, boat
shakespeare-caesar.txt: bru, caesar, brutus, haue, cassi
shakespeare-hamlet.txt: ham, haue, lord, hamlet, king
